# 06. Fixed-window start-month sweep

## Setup

In [ ]:
from pathlib import Path
import json
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").is_file():
    if root == root.parent:
        raise RuntimeError("Open this notebook inside the project repository")
    root = root.parent
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # model_aza

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from model_aza.training import start_month_sweep as sweep, transfer, tuning
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)


def frame(records):
    table = pd.DataFrame(records)
    for column in ("max_depth", "max_samples"):
        if column in table:
            table[column] = table[column].astype(str)
    return table


def show_and_save(figure, run, name):
    figure.tight_layout()
    figure.savefig(run / f"{name}.png", dpi=150)
    display(figure)
    plt.close(figure)

## Inputs

In [ ]:
quality_path = ""  # Optional quality.json from Preprocessing 01; blank uses its verified pointer
features_path = ""  # Optional features_development.json for the consistency check; blank uses preprocessing 03
start_months = list(sweep.START_MONTHS)
variants = ["interval_72", "indices_standardized_32"]
params = {**tuning.BASELINE, "n_estimators": 500}
quality_handoff, inputs = sweep.load_quality(root, quality_path or None)
print("Quality handoff:", quality_handoff)
print("Development regions:", len(inputs["regions"]))

## Build capped development rows per start month

In [ ]:
run = tuning.new_run(root, "start_month_sweep", quality_handoff, area=transfer.AREA,
                     start_months=start_months, variants=variants, params={k: str(v) for k, v in params.items()})
month_rows, coverage, exclusions = {}, [], {}
for month in start_months:
    rows, exclusions[month] = sweep.month_rows(inputs, month)
    month_rows[month] = rows
    coverage.extend(sweep.coverage(rows, month))
    print(f"start month {month}: {len(rows)} pixels, {len({r['partition_group'] for r in rows})} parcels", flush=True)
tuning.save_records(run, "coverage", coverage)
(run / "shared_location_exclusions.json").write_text(json.dumps(
    {month: {"shared_locations": value["shared_locations"], "excluded_rows": sum(r["pixel_rows"] for r in value["excluded"])}
     for month, value in exclusions.items()}, indent=2))
del inputs  # Release imagery inputs before loading the feature export

## Coverage by start month

In [ ]:
coverage_table = pd.DataFrame(coverage)
display(coverage_table.pivot_table(index="start_month", columns=["partition", "crop"], values="parcels"))
display(coverage_table.pivot_table(index="start_month", columns=["partition", "crop"], values="pixels"))

## Consistency with Preprocessing 03

In [ ]:
from crop_pipeline.windows import external
features_file, development = tuning.load_development(root, features_path or None)
pipeline_month = development.get("fixed_start_month", 1)
pipeline_rows = [r for r in development["rows"]["fixed_full"] if r["partition"] in {"train", "validation"}]
same = {external.pixel_key(r) for r in pipeline_rows} == {external.pixel_key(r) for r in month_rows.get(pipeline_month, [])}
consistency = {"features": str(features_file), "pipeline_start_month": pipeline_month,
               "pipeline_pixels": len(pipeline_rows), "sweep_pixels": len(month_rows.get(pipeline_month, [])),
               "identical_pixel_keys": same}
(run / "consistency.json").write_text(json.dumps(consistency, indent=2))
del development, pipeline_rows
consistency

## Scores by start month

In [ ]:
records, holdout, predictability = [], [], []
for month, rows in month_rows.items():
    if not sweep.has_all_crops(rows):
        records.append({"start_month": month, "status": "incomplete_crop_coverage"})
        continue
    train = [r for r in rows if r["partition"] == "train"]
    validation = [r for r in rows if r["partition"] == "validation"]
    for variant in variants:
        pooled = transfer.spatial_cv(train, params, variant)
        locked = transfer.locked_validation(train, validation, params, variant)
        scored = [r for r in transfer.leave_one_source_out(rows, params, variant) if r["status"] == "scored"]
        signal = [r for r in transfer.source_predictability(rows, variant) if r["status"] == "scored"]
        holdout.extend({"start_month": month, **r} for r in scored)
        predictability.extend({"start_month": month, **r} for r in signal)
        records.append({"start_month": month, "variant": variant, "status": "scored",
                        "spatial_macro_f1": pooled["macro_f1"], "fold_min": min(pooled["fold_macro_f1"]),
                        "fold_max": max(pooled["fold_macro_f1"]), "locked_validation_macro_f1": locked["macro_f1"],
                        "source_holdout_macro_f1_mean": sum(r["macro_f1_scored_crops"] for r in scored) / len(scored) if scored else None,
                        "source_predictability_mean": sum(r["source_balanced_accuracy"] for r in signal) / len(signal) if signal else None,
                        **{f"spatial_{crop}_f1": v["f1"] for crop, v in pooled["per_crop"].items()}})
        print(month, variant, f"spatial {pooled['macro_f1']:.3f}", flush=True)
tuning.save_records(run, "month_scores", records)
tuning.save_records(run, "month_leave_one_source_out", holdout)
tuning.save_records(run, "month_source_predictability", predictability)
display(pd.DataFrame(records).round(3))

## Scores on parcels present in every start month

In [ ]:
common = set.intersection(*({r["partition_group"] for r in rows} for rows in month_rows.values()))
print("Parcels present in every start month:", len(common))
common_records = []
for month, rows in month_rows.items():
    subset = [r for r in rows if r["partition_group"] in common]
    if not sweep.has_all_crops(subset):
        common_records.append({"start_month": month, "status": "incomplete_crop_coverage"})
        continue
    train = [r for r in subset if r["partition"] == "train"]
    validation = [r for r in subset if r["partition"] == "validation"]
    for variant in variants:
        try:
            spatial = transfer.spatial_cv(train, params, variant)["macro_f1"]
        except ValueError:
            spatial = None
        locked = transfer.locked_validation(train, validation, params, variant)["macro_f1"]
        common_records.append({"start_month": month, "variant": variant, "status": "scored", "parcels": len(
            {r["partition_group"] for r in subset}), "spatial_macro_f1": spatial, "locked_validation_macro_f1": locked})
tuning.save_records(run, "common_parcel_scores", common_records)
display(pd.DataFrame(common_records).round(3))

## Figures

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(14, 4))
coverage_table.groupby(["start_month", "crop"]).parcels.sum().unstack().plot(ax=axes[0], marker="o")
axes[0].set_ylabel("Development parcels after cap")
scores = pd.DataFrame([r for r in records if r.get("status") == "scored"])
for variant, table in scores.groupby("variant"):
    axes[1].plot(table.start_month, table.spatial_macro_f1, marker="o", label=f"{variant} spatial CV")
    axes[1].plot(table.start_month, table.locked_validation_macro_f1, marker="x", linestyle="--", label=f"{variant} validation")
axes[1].set_xlabel("Fixed window start month")
axes[1].set_ylabel("Parcel macro-F1")
axes[1].set_ylim(0, 1)
axes[1].legend(fontsize=8)
show_and_save(figure, run, "start_month_sweep")
print("Run folder:", run)